# Ancient Ambient Sound Level Comparison

This notebook compares two methods for estimating "ancient ambient" underwater noise levels:
- **Method 1 (ship-filtered)**: Excludes time windows with ship presence, using AIS-derived ship metrics
- **Method 2 (unfiltered)**: Uses rolling percentile statistics over longer windows without ship filtering

**Ship data analysis date range**: 2026-02-07 to 2026-02-13

> **Environment requirement**: This notebook requires the `orcasound` conda environment.
> Activate it before launching Jupyter: `conda activate orcasound`

In [1]:
import sys
from datetime import datetime, timedelta
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, '../src')
from orcasound_noise.analysis.partitioned_accessor import PartitionedAccessor
from orcasound_noise.utils.hydrophone import Hydrophone

In [ ]:
# Analysis parameters
HYDROPHONE = Hydrophone.ORCASOUND_LAB_DATA  # data_2.0 folder on acoustic-sandbox
ANALYSIS_END = datetime(2026, 2, 13, 23, 59, 59)
SHIP_DATA_START = datetime(2026, 2, 7)
EXTENDED_START = datetime(2026, 1, 31)
AWS_PROFILE = "ambient-sound-team"
AWS_REGION = "us-west-2"
CONFIDENCE_THRESHOLD = 0.5
COMM_BAND = (500, 15000)
PERCENTILES = [0.05, 0.10, 0.25, 0.50]
METHOD1_WINDOWS = [1, 2, 5, 7]   # days, capped by ship data availability
METHOD2_WINDOWS = [1, 2, 3, 5, 7, 10, 14]  # days, no ship data required
S3_SHIP_METRICS = "s3://acoustic-sandbox/ambient-sound-analysis/temp_ship_metrics"

# Make sure downstream S3 readers (Polars/object_store) use the intended AWS profile.
import os
os.environ["AWS_PROFILE"] = AWS_PROFILE
os.environ["AWS_REGION"] = AWS_REGION

## 1. Ship Metrics Data Access

Load pre-computed ship track metrics from S3. Data covers 2026-02-07 to 2026-02-13 and is partitioned by year/month/day.

**Known data quirk**: The `bb_series` column contains a constant value of approximately −171.64 dB for many ship passages. This equals the negated `bb_ref` constant from `Hydrophone.ORCASOUND_LAB` and indicates that the broadband acoustic series was not populated for those passages (e.g., ship was too far from the hydrophone or there was an acoustic data gap during that time). This analysis uses raw acoustic data from `PartitionedAccessor` directly, not the pre-aggregated values in the ship metrics file.

In [3]:
# Load a single day as a schema/data check
ship_sample = pl.scan_parquet(
    f"{S3_SHIP_METRICS}/year=2026/month=02/day=11/*.parquet",
    storage_options={"aws_profile": AWS_PROFILE, "aws_region": AWS_REGION}
).collect()

print(f"Shape: {ship_sample.shape}")
print(f"\nColumns ({len(ship_sample.columns)}):")
for col, dtype in zip(ship_sample.columns, ship_sample.dtypes):
    print(f"  {col}: {dtype}")
print(f"\nSample rows:")
ship_sample.head(3)

Shape: (82, 27)

Columns (27):
  id_track: Int64
  s_timestamp: Datetime(time_unit='ns', time_zone=None)
  l_timestamp: Datetime(time_unit='ns', time_zone=None)
  duration: Float64
  avg_speed: Float64
  max_speed: Float64
  min_speed: Float64
  distance: Float64
  curviness: Float64
  confidence: Float64
  min_dist: Float64
  is_isolated: Boolean
  mmsi: String
  imo: String
  name: String
  draft: Float64
  type_id: String
  type: Float64
  comm_avg: Float64
  comm_95th: Float64
  comm_75th: Float64
  comm_50th: Float64
  bb_avg: Float64
  bb_95th: Float64
  bb_50th: Float64
  comm_bb_series: List(Float64)
  bb_series: List(Float64)

Sample rows:


id_track,s_timestamp,l_timestamp,duration,avg_speed,max_speed,min_speed,distance,curviness,confidence,min_dist,is_isolated,mmsi,imo,name,draft,type_id,type,comm_avg,comm_95th,comm_75th,comm_50th,bb_avg,bb_95th,bb_50th,comm_bb_series,bb_series
i64,datetime[ns],datetime[ns],f64,f64,f64,f64,f64,f64,f64,f64,bool,str,str,str,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,list[f64],list[f64]
49141147,2026-02-11 00:13:36,2026-02-11 00:37:24,1428.0,19.592692,21.0,17.3,14.345967,1.013894,0.971613,2227.774279,false,"""nan""","""nan""",null,null,null,null,81.750882,81.750882,81.750882,81.750882,-171.640658,-171.640658,-171.640658,"[81.750882, 81.750882, … 81.750882]","[-171.640658, -171.640658, … -171.640658]"
49141180,2026-02-11 00:21:36,2026-02-11 00:22:09,33.0,5.38,6.1,2.5,0.14383,1.010279,0.942537,6017.673203,true,"""nan""","""nan""",null,null,null,null,81.750882,81.750882,81.750882,81.750882,-171.640658,-171.640658,-171.640658,"[81.750882, 81.750882, … 81.750882]","[-171.640658, -171.640658, … -171.640658]"
49141312,2026-02-11 00:55:10,2026-02-11 01:30:23,2113.0,12.891111,13.8,11.6,14.23555,1.021873,0.982034,2173.441375,false,"""nan""","""nan""",null,null,null,null,75.165244,81.750882,81.750882,71.40772,-55.743477,27.383479,18.255443,"[81.750882, 81.750882, … 71.168135]","[-171.640658, -171.640658, … 23.469339]"


In [4]:
# Verify expected columns are present
expected_cols = ['s_timestamp', 'l_timestamp', 'confidence', 'min_dist', 'is_isolated']
missing = [c for c in expected_cols if c not in ship_sample.columns]
assert not missing, f"Missing expected columns: {missing}"
assert ship_sample.shape[0] > 0, "Ship metrics sample returned 0 rows"
print(f"✓ Schema validation passed: {ship_sample.shape[0]} rows, {ship_sample.shape[1]} columns")
print(f"✓ All expected columns present: {expected_cols}")

✓ Schema validation passed: 82 rows, 27 columns
✓ All expected columns present: ['s_timestamp', 'l_timestamp', 'confidence', 'min_dist', 'is_isolated']


## 2. Acoustic Data Access via PartitionedAccessor

Verify S3 access using a 5-minute smoke test before loading the full dataset. This also validates the bug fixes applied in this sprint to `get_time_range` and `get_broadband`.

In [5]:
# Instantiate accessor (no time args in constructor)
accessor = PartitionedAccessor(HYDROPHONE)

# 5-minute broadband smoke test
t_start = datetime(2026, 2, 13, 0, 0, 0)
t_end = datetime(2026, 2, 13, 0, 5, 0)

bb_sample = accessor.get_time_range(t_start, t_end, psd=False)
print(f"Broadband sample shape: {bb_sample.shape}")
print(f"Broadband columns: {bb_sample.columns}")
print(f"Broadband '0' column dtype: {bb_sample['0'].dtype}")
print(bb_sample.head(3))

OSError: object-store error: Generic S3 error: Error performing list request: Error performing GET https://s3.us-west-2.amazonaws.com/acoustic-sandbox?list-type=2&prefix=ambient-sound-analysis%2Forcasound_lab%2Fbroadband%2Fhydrophone%3Dorcasound_lab%2F in 92.605333ms - Server returned non-2xx status code: 400 Bad Request: <?xml version="1.0" encoding="UTF-8"?>
<Error><Code>ExpiredToken</Code><Message>The provided token has expired.</Message><Token-0>IQoJb3JpZ2luX2VjENP//////////wEaCXVzLXdlc3QtMiJHMEUCIQCuJQKntyb2QqYl+aQu9/5jCDJ/adscYzom1MyT5KDWzAIgQ7WPejiUpi4qFGPpIhpxRqARckM1gCL1FBDxvOSEiCoq7AIIHBAAGgw0ODE2NjUwOTYzMjAiDEPvE+KIVB59y3HCgyrJAtObUJXPsZgp7i01GJo1HaVI0LKOePGPYvW9y83GfIYuu8QGS4VHnWsm+pgXT/lDBQVDe7aSFlsIkq9dX0Np8KvARKnxZ7hW9xEAf3Xkdm/kTfkRy7gWxaN7ITEBORMkNRQGVawzBXODt1qP8YXjpTO21Cy8rG+ZKFyBkkv6sRraWpVp83Dpa0d2e428TjmfowieZyuOzin7MZXNJR6GRx0FE5Tgb8i836KoIbSEar7qq+oaveclE3H+p7wpuM417YGxiXuzc2XEEoZcwf/aHwjY4RJS1hHFCBNLFsG0RRrd6aBwfepY0/M+Nez+gLGKhriu5aHWGTMaanZN1Vo9IYPgooRsaXr3TR

In [ ]:
# 5-minute comm band smoke test (validates the get_broadband bug fix)
comm_sample = accessor.get_broadband(t_start, t_end, COMM_BAND[0], COMM_BAND[1], ref=1)
print(f"Comm band sample shape: {comm_sample.shape}")
print(f"Comm band columns: {comm_sample.columns}")
print(comm_sample.head(3))

In [ ]:
# Validate both samples
assert bb_sample.shape[0] > 0, "Broadband returned 0 rows"
assert '0' in bb_sample.columns, "Broadband missing '0' column"
assert comm_sample.shape[0] > 0, "Comm band returned 0 rows"
assert 'sound_pressure_level_db' in comm_sample.columns, f"Comm band missing expected column; got: {comm_sample.columns}"

print("✓ Broadband access: OK")
print("✓ Comm band access (get_broadband fix validated): OK")
print(f"✓ Broadband range: {bb_sample['0'].min():.1f} to {bb_sample['0'].max():.1f} dB")
print(f"✓ Comm band range: {comm_sample['sound_pressure_level_db'].min():.1f} to {comm_sample['sound_pressure_level_db'].max():.1f} dB")

## 3. Ship Metrics Loading

In [ ]:
import os

# Ensure AWS profile is set for polars S3 access
os.environ["AWS_PROFILE"] = AWS_PROFILE

# Load all 7 days of ship metrics in a single scan_parquet call
ship_raw = pl.scan_parquet(
    f"{S3_SHIP_METRICS}/year=2026/month=*/**/*.parquet",
    storage_options={"aws_profile": AWS_PROFILE, "aws_region": AWS_REGION},
).collect()

# Parse s_timestamp and l_timestamp to Datetime if stored as strings or integers
for col in ("s_timestamp", "l_timestamp"):
    if ship_raw[col].dtype == pl.Utf8:
        ship_raw = ship_raw.with_columns(
            pl.col(col).str.to_datetime(time_unit="us").alias(col)
        )
    elif ship_raw[col].dtype in (pl.Int64, pl.Int32, pl.Float64):
        # Assume epoch seconds; cast to microseconds
        ship_raw = ship_raw.with_columns(
            (pl.col(col).cast(pl.Int64) * 1_000_000).cast(pl.Datetime("us")).alias(col)
        )

print(f"Total rows loaded: {ship_raw.shape[0]:,}")
print(f"Columns: {ship_raw.columns}")
print(f"s_timestamp dtype: {ship_raw['s_timestamp'].dtype}")
print(f"l_timestamp dtype: {ship_raw['l_timestamp'].dtype}")
print(f"Date range: {ship_raw['s_timestamp'].min()} -> {ship_raw['l_timestamp'].max()}")
ship_raw.head(3)

In [ ]:
# Apply confidence threshold filter
rows_before = ship_raw.shape[0]
ships_filtered = ship_raw.filter(pl.col("confidence") >= CONFIDENCE_THRESHOLD)
rows_after = ships_filtered.shape[0]

print(f"Rows before filter : {rows_before:,}")
print(f"Rows after filter  : {rows_after:,}  (confidence >= {CONFIDENCE_THRESHOLD})")
print(f"Rows removed       : {rows_before - rows_after:,}  ({(rows_before - rows_after) / rows_before * 100:.1f}%)")
print()
ships_filtered.head()

## 4. Ship Presence Mask

In [ ]:
# Build a per-second DataFrame spanning the full 7-day window
seconds_df = pl.DataFrame(
    {"ts": pl.datetime_range(SHIP_DATA_START, ANALYSIS_END, interval="1s", eager=True)}
)

# Mark each second as ship-present if it falls within any ship track interval
try:
    seconds_with_ships = (
        seconds_df
        .join_where(
            ships_filtered.select(["s_timestamp", "l_timestamp"]),
            pl.col("ts") >= pl.col("s_timestamp"),
            pl.col("ts") <= pl.col("l_timestamp"),
        )
        .select("ts")
        .unique()
    )
except AttributeError:
    # Fallback for polars < 0.19 (join_where not available)
    ship_present_series = pl.Series("ts", [], dtype=seconds_df["ts"].dtype)
    for row in ships_filtered.select(["s_timestamp", "l_timestamp"]).iter_rows(named=True):
        mask = (seconds_df["ts"] >= row["s_timestamp"]) & (seconds_df["ts"] <= row["l_timestamp"])
        ship_present_series = pl.concat([ship_present_series, seconds_df.filter(mask)["ts"]])
    seconds_with_ships = pl.DataFrame({"ts": ship_present_series}).unique()

# Build presence mask via left join (avoids O(n*m) is_in on large Series)
presence_mask = (
    seconds_df
    .join(
        seconds_with_ships.unique().with_columns(pl.lit(True).alias("ship_present")),
        on="ts",
        how="left"
    )
    .with_columns(pl.col("ship_present").fill_null(False))
)

total_seconds = len(presence_mask)
ship_present_count = presence_mask["ship_present"].sum()
ship_free_count = total_seconds - ship_present_count

print(f"Total seconds      : {total_seconds:,}")
print(f"Ship-present count : {ship_present_count:,}  ({ship_present_count / total_seconds * 100:.1f}%)")
print(f"Ship-free count    : {ship_free_count:,}  ({ship_free_count / total_seconds * 100:.1f}%)")
presence_mask.head(5)

## 5. Ship-Free Fraction by Window

In [ ]:
# Compute ship-free fraction for each Method 1 window length
results = []
for d in METHOD1_WINDOWS:
    window_start = ANALYSIS_END - timedelta(days=d)
    window_slice = presence_mask.filter(
        (pl.col("ts") >= window_start) & (pl.col("ts") <= ANALYSIS_END)
    )
    window_total = len(window_slice)
    ship_free_seconds = (~window_slice["ship_present"]).sum()
    ship_free_pct = ship_free_seconds / window_total if window_total > 0 else None
    results.append({
        "window_days": d,
        "total_seconds": window_total,
        "ship_free_seconds": ship_free_seconds,
        "ship_free_pct": ship_free_pct,
    })

ship_free_df = pl.DataFrame(results)
ship_free_df

## Plot C: Ship-Free Fraction by Window Length

In [ ]:
import matplotlib.pyplot as plt

# Convert to pandas for plotting
ship_free_pd = ship_free_df.to_pandas()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(ship_free_pd["window_days"], ship_free_pd["ship_free_pct"] * 100, color="steelblue", edgecolor="white")
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_xlabel("Window Length (days)")
ax.set_ylabel("Ship-Free Time (%)")
ax.set_title("Ship-Free Fraction by Analysis Window (Method 1)")
ax.set_xticks(ship_free_pd["window_days"])
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## Plot D: Ship Presence by Hour of Day

In [ ]:
hourly = (
    presence_mask
    .with_columns(pl.col("ts").dt.hour().alias("hour"))
    .group_by("hour")
    .agg(pl.col("ship_present").mean().alias("ship_present_frac"))
    .sort("hour")
)
hourly_pd = hourly.to_pandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(hourly_pd["hour"], hourly_pd["ship_present_frac"] * 100, color="salmon", edgecolor="white")
ax.set_xlabel("Hour (UTC)")
ax.set_ylabel("Fraction of Seconds with Ship Present (%)")
ax.set_title("Ship Presence by Hour of Day (Feb 7–13, 2026)")
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

## 6. Acoustic Data Loading (Full Range)

In [ ]:
%%time
# Load broadband for the full 14-day range (ending 2/13, starting 1/31 for 14-day Method 2 windows)
bb_df = accessor.get_time_range(EXTENDED_START, ANALYSIS_END, psd=False)
print(f"Broadband loaded: {bb_df.shape}, range {bb_df['__index_level_0__'].min()} to {bb_df['__index_level_0__'].max()}")

In [ ]:
%%time
# Load comm band (500–15,000 Hz) for the full range — requires reading all PSD columns, ~30-60 seconds
comm_df = accessor.get_broadband(EXTENDED_START, ANALYSIS_END, COMM_BAND[0], COMM_BAND[1], ref=1)
print(f"Comm band loaded: {comm_df.shape}")

In [ ]:
# Merge broadband and comm band into a single acoustic DataFrame
# bb_df has column '0' (broadband dB), comm_df has 'sound_pressure_level_db' (comm band dB)
acoustic_df = (
    bb_df.rename({"0": "broadband_db", "__index_level_0__": "timestamp"})
    .select(["timestamp", "broadband_db"])
    .join(
        comm_df.rename({"__index_level_0__": "timestamp", "sound_pressure_level_db": "comm_band_db"})
        .select(["timestamp", "comm_band_db"]),
        on="timestamp",
        how="inner"
    )
    .sort("timestamp")
)
print(f"Merged acoustic_df: {acoustic_df.shape}")
print(f"Date range: {acoustic_df['timestamp'].min()} to {acoustic_df['timestamp'].max()}")
print(f"Sample broadband (dB): min={acoustic_df['broadband_db'].min():.1f}, max={acoustic_df['broadband_db'].max():.1f}")
acoustic_df.head(3)

## 7. Join Ship-Presence Mask to Acoustic Data

In [ ]:
# Left-join ship-presence mask onto acoustic_df by timestamp
# Acoustic rows outside 2/7–2/13 (before ship data) get ship_present = False (null → False)
acoustic_df = (
    acoustic_df
    .join(
        presence_mask.rename({"ts": "timestamp"}),
        on="timestamp",
        how="left"
    )
    .with_columns(pl.col("ship_present").fill_null(False))
)

# Report ship presence stats for the 2/7–2/13 acoustic window
window_slice = acoustic_df.filter(
    (pl.col("timestamp") >= pl.lit(SHIP_DATA_START)) &
    (pl.col("timestamp") <= pl.lit(ANALYSIS_END))
)
ship_present_count = window_slice["ship_present"].sum()
ship_free_count = len(window_slice) - ship_present_count
print(f"Acoustic rows 2/7–2/13: {len(window_slice):,}")
print(f"  Ship-present: {ship_present_count:,} ({100*ship_present_count/len(window_slice):.1f}%)")
print(f"  Ship-free:    {ship_free_count:,} ({100*ship_free_count/len(window_slice):.1f}%)")
print(f"Total acoustic_df rows: {len(acoustic_df):,}")
acoustic_df.head(3)

## 8. Method 1 — Ship-Filtered Percentiles

In [ ]:
method1_results = []

for d in METHOD1_WINDOWS:
    window_start = ANALYSIS_END - timedelta(days=d)

    # Slice to window and filter ship-free
    window_df = acoustic_df.filter(
        (pl.col("timestamp") >= pl.lit(window_start)) &
        (pl.col("timestamp") <= pl.lit(ANALYSIS_END)) &
        (~pl.col("ship_present"))
    )

    n_seconds = len(window_df)

    if n_seconds == 0:
        print(f"Window {d}d: 0 ship-free seconds — skipping")
        continue

    # Look up ship_free_pct from ship_free_df computed in Sprint 2
    ship_free_pct_row = ship_free_df.filter(pl.col("window_days") == d)
    sfp = ship_free_pct_row["ship_free_pct"][0] if len(ship_free_pct_row) > 0 else None

    sfp_str = f"{sfp:.3f}" if sfp is not None else "N/A"
    print(f"Window {d}d: {n_seconds:,} ship-free seconds, ship_free_pct={sfp_str}")

    for band, col in [("broadband", "broadband_db"), ("comm_band", "comm_band_db")]:
        for p in PERCENTILES:
            val = window_df[col].quantile(p)
            method1_results.append({
                "method": "ship_filtered",
                "window_days": d,
                "band": band,
                "percentile": p,
                "value_db": val,
                "n_seconds": n_seconds,
                "ship_free_pct": sfp,
            })

method1_df = pl.DataFrame(method1_results, schema={
    "method": pl.String,
    "window_days": pl.Int64,
    "band": pl.String,
    "percentile": pl.Float64,
    "value_db": pl.Float64,
    "n_seconds": pl.Int64,
    "ship_free_pct": pl.Float64,
})
print(f"\nMethod 1 results: {method1_df.shape}")
method1_df.head(8)

## 9. Method 2 — Unfiltered Percentiles

In [ ]:
from datetime import timedelta

method2_results = []

for d in METHOD2_WINDOWS:
    window_start = ANALYSIS_END - timedelta(days=d)
    
    # Slice to window — no ship filter
    window_df = acoustic_df.filter(
        (pl.col("timestamp") >= pl.lit(window_start)) &
        (pl.col("timestamp") <= pl.lit(ANALYSIS_END))
    )
    
    n_seconds = len(window_df)
    
    if n_seconds == 0:
        print(f"Window {d}d: 0 rows — skipping")
        continue
    
    for band, col in [("broadband", "broadband_db"), ("comm_band", "comm_band_db")]:
        for p in PERCENTILES:
            val = window_df[col].quantile(p)
            method2_results.append({
                "method": "unfiltered",
                "window_days": d,
                "band": band,
                "percentile": p,
                "value_db": val,
                "n_seconds": n_seconds,
                "ship_free_pct": None,
            })
    
    print(f"Window {d}d: {n_seconds:,} rows")

method2_df = pl.DataFrame(method2_results, schema={
    "method": pl.String,
    "window_days": pl.Int64,
    "band": pl.String,
    "percentile": pl.Float64,
    "value_db": pl.Float64,
    "n_seconds": pl.Int64,
    "ship_free_pct": pl.Float64,
})
print(f"\nMethod 2 results: {method2_df.shape}")
method2_df.head(8)

## 10. Results Summary Table

In [ ]:
# Combine Method 1 and Method 2 results
results_df = pl.concat([method1_df, method2_df], how="vertical")
print(f"Total results: {results_df.shape} (expected 88 rows)")

# Display as pandas pivot table for readability
results_pd = results_df.to_pandas()
pivot = results_pd.pivot_table(
    index=["method", "window_days"],
    columns=["band", "percentile"],
    values="value_db",
    aggfunc="first"
)
# Format to 1 decimal place
pivot.columns = [f"{b} p{int(p*100)}" for b, p in pivot.columns]
pivot = pivot.round(1)
pivot

## Plot A: Estimated Ambient Level by Method and Window Length

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

results_pd = results_df.to_pandas()
bands = ["broadband", "comm_band"]
band_labels = {"broadband": "Broadband", "comm_band": "Comm Band (500–15k Hz)"}
colors = {0.05: "royalblue", 0.10: "seagreen", 0.25: "orange", 0.50: "crimson"}
pct_labels = {0.05: "5th pct", 0.10: "10th pct", 0.25: "25th pct", 0.50: "50th pct"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)

for ax, band in zip(axes, bands):
    m2 = results_pd[(results_pd["method"] == "unfiltered") & (results_pd["band"] == band)]
    m1 = results_pd[(results_pd["method"] == "ship_filtered") & (results_pd["band"] == band)]
    
    for p in PERCENTILES:
        color = colors[p]
        label = pct_labels[p]
        
        # Method 2: solid line
        d2 = m2[m2["percentile"] == p].sort_values("window_days")
        ax.plot(d2["window_days"], d2["value_db"], color=color, linewidth=2,
                label=f"{label} (unfiltered)")
        
        # Method 1: dashed line with markers
        d1 = m1[m1["percentile"] == p].sort_values("window_days")
        ax.plot(d1["window_days"], d1["value_db"], color=color, linewidth=2,
                linestyle="--", marker="o", markersize=6,
                label=f"{label} (ship-filtered)")
    
    ax.set_title(band_labels[band])
    ax.set_xlabel("Window Length (days)")
    ax.set_ylabel("Estimated Ancient Ambient (dB)")
    ax.set_xticks([1, 2, 3, 5, 7, 10, 14])
    ax.grid(True, alpha=0.3)

# Single legend outside the plots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.12))
fig.suptitle("Estimated Ancient Ambient Level by Method and Window Length", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Plot B: 5th Percentile Ancient Ambient — Ship-Filtered vs Unfiltered

In [ ]:
shared_windows = [1, 2, 5, 7]  # Windows where both methods have data
target_pct = 0.05

results_pd_local = results_df.to_pandas()
bands = ["broadband", "comm_band"]
band_labels = {"broadband": "Broadband", "comm_band": "Comm Band (500–15k Hz)"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=False)
bar_width = 0.35
x = np.arange(len(shared_windows))

for ax, band in zip(axes, bands):
    m1_vals = []
    m2_vals = []
    
    for d in shared_windows:
        v1 = results_pd_local[
            (results_pd_local["method"] == "ship_filtered") &
            (results_pd_local["band"] == band) &
            (results_pd_local["percentile"] == target_pct) &
            (results_pd_local["window_days"] == d)
        ]["value_db"].values
        m1_vals.append(v1[0] if len(v1) > 0 else float("nan"))
        
        v2 = results_pd_local[
            (results_pd_local["method"] == "unfiltered") &
            (results_pd_local["band"] == band) &
            (results_pd_local["percentile"] == target_pct) &
            (results_pd_local["window_days"] == d)
        ]["value_db"].values
        m2_vals.append(v2[0] if len(v2) > 0 else float("nan"))
    
    bars1 = ax.bar(x - bar_width/2, m1_vals, bar_width, label="Ship-Filtered (Method 1)", color="steelblue")
    bars2 = ax.bar(x + bar_width/2, m2_vals, bar_width, label="Unfiltered (Method 2)", color="salmon")
    
    # Annotate dB difference
    for i, (v1, v2) in enumerate(zip(m1_vals, m2_vals)):
        if not (np.isnan(v1) or np.isnan(v2)):
            diff = v2 - v1
            ymax = max(v1, v2)
            ax.annotate(f"Δ{diff:+.1f}dB", xy=(x[i], ymax), ha="center", va="bottom", fontsize=8)
    
    ax.set_title(band_labels[band])
    ax.set_xlabel("Window Length (days)")
    ax.set_ylabel("5th Percentile Level (dB)")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{d}d" for d in shared_windows])
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("5th Percentile Ancient Ambient: Ship-Filtered vs Unfiltered", fontsize=14)
plt.tight_layout()
plt.show()

## 11. Interpretation and Findings

*Fill in after running the notebook end-to-end with valid AWS credentials.*

### Key Questions This Analysis Addresses

**Does ship filtering change the ambient estimate?**  
Compare the Δ values in Plot B. If Method 1 (ship-filtered) gives systematically lower 5th-percentile estimates than Method 2 (unfiltered) by more than ~1–2 dB, ship activity meaningfully biases the unfiltered ambient baseline. A small Δ (< 1 dB) suggests ship filtering provides little additional value at this location/period.

**Does the estimate stabilize with longer windows?**  
In Plot A, look for convergence: if the lines flatten out by ~5–7 days, shorter windows are adequate. If values continue drifting with longer windows, more data is needed for a stable estimate.

**Which band is more sensitive to ship filtering?**  
Ships produce broadband noise, but the orca communication band (500–15,000 Hz) may show stronger ship contamination. Compare the Δ values in Plot B across the two subplots.

**Recommended method and window length:**  
Based on the analysis:
- If ship-filtering changes the 5th percentile by > 1 dB consistently: **use Method 1 (ship-filtered)** with the longest available window (7 days for the current ship data period).
- If ship-filtering has negligible effect: **use Method 2 (unfiltered)** with a 7–14 day window, which provides more stable estimates from a larger sample.

### Data Notes

- Ship metrics cover 2026-02-07 to 2026-02-13 (7 days). Method 1 is limited to this period.
- Acoustic data is loaded from 2026-01-31 to 2026-02-13, enabling Method 2 windows up to 14 days.
- Ship confidence threshold: ≥ 0.5. Tracks below this threshold are excluded from the ship-free mask.
- The `bb_series` column in ship metrics shows constant −171.64 dB for many ships (= negated `bb_ref` constant), indicating missing acoustic coverage during those passages. This analysis uses raw PartitionedAccessor data directly and does not rely on ship-metrics acoustic values.